# 03 · Rain and temperature: autumn rain, heat at flowering, frost, dry spells

**Field Shift** (NASA Space Apps 2026), data notebook 3 of 6. *Code written with AI assistance (Claude, Anthropic); run and checked by the team.* Needs notebook 01's files in the same Drive folder.

| Output | What's in it |
|---|---|
| `data/field.json` (adds `weather`) | The demo field: monthly rain from five sources and temperature from two since 2000, each season's autumn rain, rain start, spring dry spell, hot days at flowering and frost, plus how far the sources agree |
| `data/cells_weather.json` | For every farmland square: the same indicators as low / typical / high, monthly climate, and each season's rain and heat since 2001 |

**Sources.** Rain: CHIRPS (satellite plus stations, 5 km), NASA IMERG (satellites, 10 km), ERA5-Land (weather model, 9 km); for the field also JAXA GSMaP (satellites, 10 km) and NASA POWER (50 km). Temperature: ERA5-Land; NASA POWER as a check at the field. Rain values shown to farmers are the **median of the sources**, with their range kept for the evidence panel. Coarse data are copied into the 1 km squares they cover.

**How to run (about 5–10 minutes):** put your project ID below, then **Runtime → Run all**.

In [ ]:
EE_PROJECT = "your-cloud-project-id"    # Earth Engine Cloud project (change it if someone else runs this)
DRIVE_FOLDER = "field-shift"            # folder for this notebook's files (in Google Drive if USE_DRIVE is True)
USE_DRIVE = False                       # False: you pick the input files with an upload button. True: files are shared through Google Drive
REPO = ""                               # optional, e.g. "team-owner/field-shift": used only if a file isn't in Drive
OPTIONAL_INPUTS = ["station_temps.json"]    # optional: monthly station temperatures to correct ERA5-Land at the demo field (never goes in the repo)

## Setup

In [ ]:
import sys, subprocess
try:
    import pyproj, shapely                  # normally already in Colab
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyproj", "shapely"], check=True)

import json, math, re, time, random, hashlib, urllib.request, datetime as dt
import concurrent.futures as cf
from pathlib import Path

import ee
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely.geometry import shape

if EE_PROJECT == "your-cloud-project-id":
    raise ValueError("Put your Earth Engine Cloud project ID into EE_PROJECT (first code cell), then run again.")
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

IN_COLAB = "google.colab" in sys.modules
BASE = Path(DRIVE_FOLDER)
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError(
            f"Google Drive didn't connect ({str(e)[:80]}).\n"
            "Run this cell again. In the Google window, pick the same account and tick 'Select all' "
            "(the Google Drive box) before pressing Continue.\n"
            "If Drive can't be used at all, set USE_DRIVE = False in the first cell. The notebook will then ask you to "
            "upload notebook 01's files, and you upload its downloads to the repo yourself.") from None
    BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
elif IN_COLAB:
    BASE = Path("/content") / DRIVE_FOLDER
DATA, CACHE = BASE / "data", BASE / "cache"
for folder in (DATA, CACHE):
    folder.mkdir(parents=True, exist_ok=True)
print("Working folder:", BASE)

TO_LAEA = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
CELL = 1000
RUN_AT = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
BASELINE = (2001, 2020)          # the "normal" period: at least 15 years, printed in every file

def with_retries(fn, tries=6):
    """Earth Engine sometimes answers 'Too many requests' or times out: wait, then try again."""
    for k in range(tries):
        try:
            return fn()
        except Exception as e:
            if k == tries - 1:
                raise
            print("  retrying after:", str(e)[:100])
            time.sleep(min(60, 2 ** k + random.random()))

CHECKS = []
def check(ok, text, warn=False, fail=None):
    """Record and print a check: ✅ passed, ⚠️ worth a look, ❌ needs fixing (fail = what to print if it didn't pass)."""
    mark = "✅" if ok else ("⚠️" if warn else "❌")
    line = f"{mark} {text if ok or not fail else fail}"
    CHECKS.append(line)
    print(line)

def num(v, nd):
    """Round a value for the JSON files; empty or missing values become null."""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    if math.isnan(v) or math.isinf(v):
        return None
    return int(round(v)) if nd == 0 else round(v, nd)

def lth(values, nd, min_n=15):
    """[low, typical, high] = 10th percentile, median, 90th percentile; null if fewer than min_n values."""
    v = pd.Series(values, dtype="float64").dropna()
    if len(v) < min_n:
        return [None, None, None]
    return [num(v.quantile(0.1), nd), num(v.median(), nd), num(v.quantile(0.9), nd)]

INPUTS = ["grid.geojson", "field.json"] + list(globals().get("OPTIONAL_INPUTS", []))
missing = [n for n in ["grid.geojson", "field.json"] if not (DATA / n).exists()]
if missing and IN_COLAB and not REPO:
    from google.colab import files
    print("Choose these files (downloaded from earlier notebooks) in the upload box below, all at once:")
    print("   " + ", ".join(INPUTS) + ("   (the last ones only if you have them)" if len(INPUTS) > 2 else ""))
    for fname, content in files.upload().items():
        clean = re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)        # "field (1).json" -> "field.json"
        (DATA / clean).write_bytes(content)
        print("   got", clean)

def load_optional(name):
    """A data file if it's in the working folder, else an empty dict (no questions asked)."""
    path = DATA / name
    return json.loads(path.read_text()) if path.exists() else {}

def write_field_section(part, sections, sources):
    """This notebook's part of the demo field goes in its own file, field_<part>.json; notebook 06 merges them into field.json."""
    doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                    "field_key": FIELD_KEY, "sources": sources}}
    doc.update(sections)
    return write_json(DATA / f"field_{part}.json", doc)

def load_data_file(name):
    """A data file from the working folder, or else from the repo on GitHub (if REPO is set)."""
    path = DATA / name
    if path.exists():
        return json.loads(path.read_text())
    if REPO:
        url = f"https://raw.githubusercontent.com/{REPO}/main/data/{name}"
        with urllib.request.urlopen(url, timeout=60) as r:
            text = r.read().decode()
        path.write_text(text)
        return json.loads(text)
    if IN_COLAB:
        from google.colab import files
        print(f"{name} isn't in {DATA}. Choose it in the upload box below (you can pick several of notebook 01's files at once).")
        for fname, content in files.upload().items():
            clean = re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)       # "field (1).json" -> "field.json"
            (DATA / clean).write_bytes(content)
        if path.exists():
            return json.loads(path.read_text())
    raise FileNotFoundError(f"{name} isn't in {DATA}. Run notebook 01 first (same Google account and DRIVE_FOLDER), "
                            "or set REPO so the file can be fetched from GitHub.")

def write_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False))
    return path.stat().st_size / 1e6

GRID = load_data_file("grid.geojson")
FIELD_DOC = load_data_file("field.json")
FIELD = FIELD_DOC["field"]
SQUARES = [f["properties"] for f in GRID["features"]]
for sq in SQUARES:            # lower-left corner in metres, read from the EEA code, e.g. 1kmE4836N1990
    e, n = re.match(r"1kmE(\d+)N(\d+)", sq["id"]).groups()
    sq["x0"], sq["y0"] = int(e) * 1000, int(n) * 1000
FARM = [sq for sq in SQUARES if sq.get("farm")]
centre = shape(FIELD["outline"]).centroid                 # the field's data are worked out at the centre of its outline
FIELD_LON, FIELD_LAT = round(centre.x, 6), round(centre.y, 6)
FIELD_KEY = hashlib.md5(json.dumps(FIELD["outline"], sort_keys=True).encode()).hexdigest()[:8]   # changes if the outline is redrawn
FIELD_PT = ee.Geometry.Point([FIELD_LON, FIELD_LAT])
FIELD_GEOM = ee.Geometry(FIELD["outline"])
FIELD_AREA_M2 = FIELD["area_ha"] * 1e4

WC = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
CLC = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")
RAINFED = WC.eq(40).And(CLC.eq(211))          # the project's rain-fed arable mask (10 m)

def square(sq):
    return ee.Geometry.Rectangle([sq["x0"], sq["y0"], sq["x0"] + CELL, sq["y0"] + CELL], "EPSG:3035", False)

def squares_fc(batch):
    return ee.FeatureCollection([ee.Feature(square(sq), {"id": sq["id"]}) for sq in batch])

print(f"{len(SQUARES):,} grid squares ({len(FARM):,} farmland); demo field in {FIELD['square']} "
      f"({FIELD['area_ha']} ha, outline {FIELD['outline_source'].split(':')[0]})")

## 1. Settings
Windows used for durum wheat, the baseline crop. Monthly numbers are also saved, so other crops can use their own months.

In [ ]:
NOTEBOOK = "notebooks/03_weather.ipynb"
TODAY = dt.date.today()
LAST_SEASON = TODAY.year if TODAY.month >= 9 else TODAY.year - 1
SEASONS = list(range(2001, LAST_SEASON + 1))
START = "2000-09-01"
END = (TODAY + dt.timedelta(days=1)).isoformat()

HOT = 30.0            # °C: heat above about 30 °C at flowering and grain fill cuts durum yield (Groli et al. 2024)
FROST = 0.0           # °C: frost night
FLOWER = ((4, 15), (6, 15))   # durum flowering and grain fill, about mid-April to mid-June
WET_DAY = 1.0         # mm: a day with less than this is dry
ONSET_MM = 20.0       # mm in 3 days after 1 September: enough rain to sow

ERA5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
CHIRPS = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
IMERG = ee.ImageCollection("NASA/GPM_L3/IMERG_MONTHLY_V07")
GSMAP = ee.ImageCollection("JAXA/GPM_L3/GSMaP/v8/operational").select("hourlyPrecipRateGC")
PROJ = {"era5": ERA5.first().select("temperature_2m_max").projection(),
        "chirps": CHIRPS.first().select("precipitation").projection(),
        "imerg": IMERG.first().select("precipitation").projection()}

SOURCES = {
    "era5_land": {"name": "ERA5-Land daily aggregates (ECMWF, Copernicus Climate Change Service)", "ee_id": "ECMWF/ERA5_LAND/DAILY_AGGR",
                  "res_m": 11132, "licence": "Copernicus licence (free, with attribution)",
                  "cite": "Muñoz Sabater, J. (2019). ERA5-Land hourly data from 1950 to present. C3S Climate Data Store. doi:10.24381/cds.e2161bac"},
    "chirps": {"name": "CHIRPS daily rainfall v2.0 (Climate Hazards Center, UCSB)", "ee_id": "UCSB-CHG/CHIRPS/DAILY", "res_m": 5566,
               "licence": "Public domain (CC0)", "cite": "Funk, C. et al. (2015). Scientific Data 2: 150066. doi:10.1038/sdata.2015.66"},
    "imerg": {"name": "NASA GPM IMERG Final monthly V07", "ee_id": "NASA/GPM_L3/IMERG_MONTHLY_V07", "res_m": 11132,
              "licence": "NASA data: no restrictions on use",
              "cite": "Huffman, G. J. et al. (2023). GPM IMERG Final Precipitation L3 1 month 0.1 degree V07. NASA GES DISC. doi:10.5067/GPM/IMERG/3B-MONTH/07"},
    "gsmap": {"name": "JAXA GSMaP operational v8 (gauge-adjusted)", "ee_id": "JAXA/GPM_L3/GSMaP/v8/operational", "res_m": 11132,
              "licence": "JAXA terms of use (free, with credit)", "cite": "GSMaP data were provided by JAXA (Kubota, T. et al. 2020, Adv. Global Change Res. 67)"},
    "power": {"name": "NASA POWER daily (MERRA-2 based): rain, temperature and soil wetness, for checks at the demo field only", "ee_id": "API: power.larc.nasa.gov", "res_m": 55000,
              "licence": "NASA data: no restrictions on use", "cite": "NASA Langley Research Center POWER Project, https://power.larc.nasa.gov"},
}
print(f"Seasons {SEASONS[0]}-{SEASONS[-1]}; normal = {BASELINE[0]}-{BASELINE[1]}")

## 2. Which source pixel covers each square
Each source has its own grid (5–10 km). The notebook finds the pixel that contains each square's centre, so each pixel's record is fetched only once.

In [ ]:
FIELD_POINT = {"id": "field", "lon": FIELD_LON, "lat": FIELD_LAT}
POINTS = FARM + [FIELD_POINT]

def pixel_map(src, points):
    pts = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([p["lon"], p["lat"]]), {"id": p["id"]}) for p in points])
    ll = ee.Image.pixelLonLat().reproject(PROJ[src])
    fc = ll.sampleRegions(collection=pts, projection=PROJ[src], geometries=False)
    return {f["properties"]["id"]: (round(f["properties"]["longitude"], 4), round(f["properties"]["latitude"], 4))
            for f in with_retries(lambda: fc.getInfo())["features"]}

def cached_pixels(path, points, src):
    if path.exists():
        return {k: tuple(v) for k, v in json.loads(path.read_text()).items()}
    m = pixel_map(src, points)
    path.write_text(json.dumps(m))
    return m

PIX = {}
for src in PROJ:
    PIX[src] = cached_pixels(CACHE / f"03_pixels_{src}_farm.json", FARM, src)
    PIX[src].update(cached_pixels(CACHE / f"03_pixels_{src}_field_{FIELD_KEY}.json", [FIELD_POINT], src))
STATION = load_optional("station_temps.json")       # optional: a nearby weather station, to correct ERA5-Land at the demo field (not used in our runs: no station file)
if STATION:
    PIX["era5"].update(cached_pixels(CACHE / f"03_pixels_era5_station_{STATION['lon']}_{STATION['lat']}.json",
                                     [{"id": "station", "lon": STATION["lon"], "lat": STATION["lat"]}], "era5"))
    print(f"Station file found: {STATION['name']} ({STATION['elev_m']} m), {len(STATION['months'])} months")
UNIQUE = {src: sorted(set(m.values())) for src, m in PIX.items()}
print({src: f"{len(u)} pixels" for src, u in UNIQUE.items()})
check(all(all(p["id"] in PIX[s] for p in POINTS) for s in PIX), f"Every square matched to a pixel in each source ({len(POINTS):,} points)")

## 3. Fetch the records
One request per source per year. Each year is saved to Drive, so a rerun is quick.

In [ ]:
def pixel_fc(src):
    return ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lon, lat]), {"id": f"{lon},{lat}"}) for lon, lat in UNIQUE[src]])

def daily_stack(col, bands, names):
    def rn(img):
        d = img.date().format("YYYYMMdd")
        return img.select(bands).rename([ee.String(n + "_d").cat(d) for n in names])
    return col.map(rn).toBands()

def fetch(src, y):
    path = CACHE / f"03_{src}_{y}.csv"
    if path.exists():
        df = pd.read_csv(path, index_col="id")
        if set(df.index) >= {f"{lon},{lat}" for lon, lat in UNIQUE[src]}:     # saved copy covers every pixel needed
            return df
    a = max(f"{y}-01-01", START)
    b = min(f"{y + 1}-01-01", END)
    col = {"era5": ERA5, "chirps": CHIRPS, "imerg": IMERG}[src].filterDate(a, b)
    if with_retries(lambda: col.size().getInfo()) == 0:        # this source hasn't published that year yet
        print(f"  no {src} data for {y} yet: skipped")
        return pd.DataFrame(index=pd.Index([f"{lon},{lat}" for lon, lat in UNIQUE[src]], name="id"))
    if src == "era5":
        img = daily_stack(col, ["temperature_2m_max", "temperature_2m_min", "total_precipitation_sum"], ["tmax", "tmin", "rain"])
    elif src == "chirps":
        img = daily_stack(col, ["precipitation"], ["rain"])
    else:   # imerg monthly
        img = col.map(lambda i: i.select("precipitation").rename(ee.String("rate_m").cat(i.date().format("YYYYMM")))).toBands()
    out = img.reduceRegions(collection=pixel_fc(src), reducer=ee.Reducer.first())
    rows = []
    for f in with_retries(lambda: out.getInfo())["features"]:
        p = f["properties"]
        row = {"id": p["id"]}
        for k, v in p.items():
            m = re.search(r"(tmax|tmin|rain|rate)_([dm])(\d{6,8})$", k)
            if m:
                row[f"{m.group(1)}_{m.group(3)}"] = v
        rows.append(row)
    df = pd.DataFrame(rows).set_index("id")
    df.to_csv(path)
    return df

jobs = [(s, y) for s in ("era5", "chirps", "imerg") for y in range(2000, TODAY.year + 1)]
t0, raw = time.time(), {}
with cf.ThreadPoolExecutor(max_workers=8) as pool:
    futs = {pool.submit(fetch, s, y): (s, y) for s, y in jobs}
    for i, fut in enumerate(cf.as_completed(futs), 1):
        raw[futs[fut]] = fut.result()
        if i % 10 == 0 or i == len(futs):
            print(f"  {i}/{len(futs)} requests done, {time.time() - t0:.0f} s")

def long_table(src, var):
    """pixel x date table for one variable."""
    frames = [raw[(src, y)].filter(regex=f"^{var}_") for y in range(2000, TODAY.year + 1) if (src, y) in raw]
    df = pd.concat(frames, axis=1)
    last = max((c.split("_")[1] for c in df.columns), default="none")
    print(f"  {src} {var}: data up to {last}")
    fmt = "%Y%m" if var == "rate" else "%Y%m%d"
    df.columns = pd.to_datetime([c.split("_")[1] for c in df.columns], format=fmt)
    return df.T.sort_index()                      # rows = dates, columns = pixels

daily = {"era5_tmax": long_table("era5", "tmax") - 273.15, "era5_tmin": long_table("era5", "tmin") - 273.15,
         "era5_rain": long_table("era5", "rain") * 1000, "chirps_rain": long_table("chirps", "rain")}
rate = long_table("imerg", "rate")
imerg_month = rate.mul(rate.index.days_in_month * 24, axis=0)       # mm per hour -> mm per month

def use_nearest_land(src, valid):
    """Squares on the coast can sit in a pixel that is sea, with no data: point them at the nearest pixel that has data."""
    good = [tuple(float(x) for x in k.split(",")) for k in valid]
    stuck = [pid for pid, (lon, lat) in PIX[src].items() if f"{lon},{lat}" not in valid]
    for pid in stuck:
        lon, lat = PIX[src][pid]
        near = min(good, key=lambda g: (g[0] - lon) ** 2 + (g[1] - lat) ** 2) if good else None
        if near and math.hypot(near[0] - lon, near[1] - lat) <= 0.15:          # at most about one pixel away
            PIX[src][pid] = near
    left = sum(1 for lon, lat in PIX[src].values() if f"{lon},{lat}" not in valid)
    return len(stuck), left

stuck, left = use_nearest_land("era5", {c for c in daily["era5_tmax"].columns if daily["era5_tmax"][c].notna().any()})
check(left == 0, f"ERA5-Land: {stuck} squares fall in sea pixels and use the nearest land pixel", warn=True,
      fail=f"ERA5-Land: {left} squares are in sea pixels with no land pixel nearby, so they have no ERA5-Land values")
print({k: f"{v.shape[0]} days x {v.shape[1]} pixels" for k, v in daily.items()}, f"IMERG {imerg_month.shape[0]} months")

## 4. The demo field: GSMaP and NASA POWER as well

In [ ]:
def gsmap_year(y):
    path = CACHE / f"03_gsmap_{FIELD_KEY}_{y}.csv"
    if path.exists():
        return pd.read_csv(path)
    bands = []
    for m in range(1, 13):
        m0 = dt.date(y, m, 1)
        if m0 > TODAY:
            break
        m1 = dt.date(y + (m == 12), m % 12 + 1, 1)
        c = GSMAP.filterDate(m0.isoformat(), min(m1, TODAY).isoformat())
        total = ee.Image(ee.Algorithms.If(c.size().gt(0), c.sum(), ee.Image.constant(-999)))
        bands += [total.rename(f"rain_{m0:%Y%m}"), ee.Image.constant(c.size()).rename(f"hours_{m0:%Y%m}")]
    vals = with_retries(lambda: ee.Image.cat(bands).reduceRegion(ee.Reducer.first(), FIELD_PT, 11132).getInfo())
    rows = [{"month": k.split("_")[1], "rain": vals.get(k), "hours": vals.get("hours_" + k.split("_")[1])}
            for k in vals if k.startswith("rain_")]
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df

with cf.ThreadPoolExecutor(max_workers=6) as pool:
    gs = pd.concat(list(pool.map(gsmap_year, range(2014, TODAY.year + 1))), ignore_index=True)
gs["date"] = pd.to_datetime(gs["month"].astype(str), format="%Y%m")
gs = gs.set_index("date").sort_index()
full = gs["hours"] >= 0.9 * 24 * gs.index.days_in_month
gsmap_month = gs["rain"].where(full & (gs["rain"] >= 0))
first = gsmap_month.first_valid_index()
print(f"GSMaP at the field: {gsmap_month.notna().sum()} complete months" + (f" since {first:%b %Y}" if first is not None else ""))

def power_daily():
    path = CACHE / f"03_power_field_{FIELD_KEY}.csv"
    if path.exists():
        return pd.read_csv(path, index_col=0, parse_dates=True)
    url = ("https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M_MAX,T2M_MIN,PRECTOTCORR&community=AG"
           f"&longitude={FIELD_LON}&latitude={FIELD_LAT}&start={START.replace('-', '')}"
           f"&end={(TODAY - dt.timedelta(days=1)):%Y%m%d}&format=JSON")
    with urllib.request.urlopen(url, timeout=180) as r:
        data = json.loads(r.read().decode())
    df = pd.DataFrame(data["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df = df.replace(data.get("header", {}).get("fill_value", -999), np.nan).sort_index()
    df.to_csv(path)
    return df

try:
    power = power_daily()
    check(len(power) > 9000, f"NASA POWER at the field: {len(power):,} days")
except Exception as e:
    power = None
    check(False, f"NASA POWER not reached ({str(e)[:80]}); the field uses the other sources", warn=True)

## 5. Season numbers
For each source pixel and season: autumn rain (October–November), season rain (October–May), spring rain (March–May), when the autumn rain starts (first 3 days with 20 mm after 1 September), the longest spring dry spell, hot days (30 °C or more) at flowering, and frost nights. Then the same by month.

In [ ]:
def season_of(d):
    return d.year + (d.month >= 9)

def window(series, season, m0d0, m1d1):
    """Values between two (month, day) points of a season; months 9-12 belong to the previous calendar year."""
    def ts(md):
        m, d = md
        return pd.Timestamp(season - 1 if m >= 9 else season, m, d)
    return series[(series.index >= ts(m0d0)) & (series.index < ts(m1d1))]

def longest_run(flags):
    best = run = 0
    for f in flags:
        run = run + 1 if f else 0
        best = max(best, run)
    return best

def rain_seasons(daily_rain):
    """{season: {...}} from a daily rain series in mm."""
    out = {}
    for s in SEASONS:
        yr = daily_rain[(daily_rain.index >= pd.Timestamp(s - 1, 9, 1)) & (daily_rain.index < pd.Timestamp(s, 9, 1))]
        if yr.notna().sum() < 0.95 * len(yr) or len(yr) < 360:
            continue
        aut, sea, spr = window(yr, s, (10, 1), (12, 1)), window(yr, s, (10, 1), (6, 1)), window(yr, s, (3, 1), (6, 1))
        three = yr.rolling(3).sum()
        hit = three[three >= ONSET_MM]
        out[s] = {"rain_autumn": aut.sum(), "rain_season": sea.sum(), "rain_spring": spr.sum(), "rain_year": yr.sum(),
                  "rain_onset_day": (hit.index[0] - pd.Timestamp(s - 1, 9, 1)).days if len(hit) else np.nan,
                  "dry_spell_spring": longest_run((spr.fillna(0) < WET_DAY).to_numpy())}
    return out

def monthly_rain_seasons(month_rain):
    """{season: {...}} from monthly totals (IMERG, GSMaP)."""
    out = {}
    for s in SEASONS:
        yr = month_rain[(month_rain.index >= pd.Timestamp(s - 1, 9, 1)) & (month_rain.index < pd.Timestamp(s, 9, 1))]
        if yr.notna().sum() < 12:
            continue
        pick = lambda months: yr[yr.index.month.isin(months)].sum()
        out[s] = {"rain_autumn": pick([10, 11]), "rain_season": pick([10, 11, 12, 1, 2, 3, 4, 5]),
                  "rain_spring": pick([3, 4, 5]), "rain_year": yr.sum()}
    return out

def temp_seasons(tmax, tmin):
    out = {}
    for s in SEASONS:
        fl, wi = window(tmax, s, *FLOWER), window(tmin, s, (12, 1), (3, 1))
        late = window(tmin, s, (3, 1), (5, 1))
        if fl.notna().sum() < 55 or wi.notna().sum() < 85:
            continue
        out[s] = {"hot_days_flower": int((fl >= HOT).sum()), "tmax_flower": fl.mean(),
                  "frost_days_winter": int((wi <= FROST).sum()), "late_frost_days": int((late <= FROST).sum())}
    return out

def month_climate_from(month_values):
    """12 x [low, typical, high] over the normal years from a monthly series."""
    m = month_values[(month_values.index.year >= BASELINE[0]) & (month_values.index.year <= BASELINE[1])]
    return [lth(m[m.index.month == k].values, 1) for k in range(1, 13)]

def to_month(daily_series, how):
    g = daily_series.resample("MS")
    out = g.sum(min_count=25) if how == "sum" else g.mean()
    return out[g.count() >= 25]

# per pixel tables
R, T, M = {}, {}, {}
for key in ("era5_rain", "chirps_rain"):
    src = key.split("_")[0]
    for pix in daily[key].columns:
        R[(src, pix)] = rain_seasons(daily[key][pix])
        M[(src, pix)] = to_month(daily[key][pix], "sum")
for pix in imerg_month.columns:
    R[("imerg", pix)] = monthly_rain_seasons(imerg_month[pix])
    M[("imerg", pix)] = imerg_month[pix]
for pix in daily["era5_tmax"].columns:
    T[pix] = temp_seasons(daily["era5_tmax"][pix], daily["era5_tmin"][pix])
print(f"Season tables: {len(R)} rain records, {len(T)} temperature records")

In [ ]:
RAIN_KEYS = ["rain_autumn", "rain_season", "rain_spring", "rain_year", "rain_onset_day", "dry_spell_spring"]
TEMP_KEYS = ["hot_days_flower", "tmax_flower", "frost_days_winter", "late_frost_days"]

def pix_key(src, point_id):
    lon, lat = PIX[src][point_id]
    return f"{lon},{lat}"

def combine_point(point_id):
    """Season table and monthly climate for one square (or the field), using the median of the rain sources."""
    keys = {s: pix_key(s, point_id) for s in PROJ}
    tables = {s: R.get((s, keys[s]), {}) for s in PROJ}
    seasons = {}
    for season in SEASONS:
        row = {}
        for k in RAIN_KEYS:
            vals = [tables[s][season][k] for s in PROJ if season in tables[s] and k in tables[s][season]]
            vals = [v for v in vals if v is not None and np.isfinite(v)]
            row[k] = float(np.median(vals)) if vals else np.nan
        row.update(T.get(keys["era5"], {}).get(season, {}))
        seasons[season] = row
    month_rain = pd.concat([M[(s, keys[s])].rename(s) for s in PROJ if (s, keys[s]) in M], axis=1).median(axis=1)
    tx, tn = daily["era5_tmax"][keys["era5"]], daily["era5_tmin"][keys["era5"]]
    clim = {"rain_month": month_climate_from(month_rain), "tmax_month": month_climate_from(to_month(tx, "mean")),
            "tmin_month": month_climate_from(to_month(tn, "mean")),
            "hot30_month": month_climate_from((tx >= HOT).astype(float).where(tx.notna()).resample("MS").sum(min_count=25)),
            "frost_month": month_climate_from((tn <= FROST).astype(float).where(tn.notna()).resample("MS").sum(min_count=25))}
    by_source = {s: lth([v["rain_year"] for y, v in tables[s].items() if BASELINE[0] <= y <= BASELINE[1]], 0)[1] for s in PROJ}
    return seasons, clim, by_source

def typical_of(seasons, key, nd):
    return lth([v.get(key) for y, v in seasons.items() if BASELINE[0] <= y <= BASELINE[1]], nd)

ND = {"rain_autumn": 0, "rain_season": 0, "rain_spring": 0, "rain_year": 0, "rain_onset_day": 0, "dry_spell_spring": 0,
      "hot_days_flower": 0, "tmax_flower": 1, "frost_days_winter": 0, "late_frost_days": 0}
# Squares that share the same source pixels share one record, which keeps the file small
combo_key, records, cells_out = {}, {}, {}
for sq in FARM:
    combo = tuple(PIX[s][sq["id"]] for s in PROJ)
    if combo not in combo_key:
        seasons, clim, by_source = combine_point(sq["id"])
        rec = dict(clim)
        rec.update({k: typical_of(seasons, k, nd) for k, nd in ND.items()})
        rec["rain_year_by_source"] = by_source
        for k in ("rain_season", "rain_autumn", "hot_days_flower"):
            rec[f"{k}_by_season"] = [num(seasons[s].get(k), ND[k]) for s in SEASONS]
        combo_key[combo] = f"w{len(records)}"
        records[combo_key[combo]] = rec
    cells_out[sq["id"]] = combo_key[combo]
print(f"Weather for {len(cells_out):,} squares from {len(records)} distinct pixel combinations")
check(len(cells_out) == len(FARM), f"Weather numbers for all {len(FARM):,} farmland squares")

## 6. The demo field in detail, and the checks

In [ ]:
f_seasons, f_clim, f_by_source = combine_point("field")
fkeys = {s: pix_key(s, "field") for s in PROJ}

# Quality control for GSMaP and POWER (comparison only): drop months that are impossible next to the main three sources
main_month = pd.concat([M[(s, fkeys[s])].rename(s) for s in PROJ if (s, fkeys[s]) in M], axis=1).median(axis=1)
def implausible(month_series):
    ref = main_month.reindex(month_series.index)
    return (month_series < 0) | (month_series > np.maximum(3 * ref, ref + 200))
bad_g = implausible(gsmap_month)
gsmap_month = gsmap_month.mask(bad_g)
bad_p_months = []
if power is not None:
    pm = to_month(power["PRECTOTCORR"], "sum")
    bad_p = implausible(pm)
    bad_p_months = list(pm.index[bad_p])
    in_bad = power.index.to_period("M").to_timestamp().isin(bad_p_months)
    power.loc[in_bad, "PRECTOTCORR"] = np.nan
removed = [f"GSMaP {d:%b %Y}" for d in gsmap_month.index[bad_g]] + [f"POWER {d:%b %Y}" for d in bad_p_months]
check(len(removed) <= 6, f"Comparison sources: {len(removed)} impossible months removed" + (f" ({', '.join(removed)})" if removed else ""),
      warn=True)

# Optional: correct ERA5-Land temperature at the demo field with a nearby weather station, month by month. Not used in our runs: the station records could not be downloaded, so there was no station file and no correction was applied (the app keeps its cool-temperature caveat)
LAPSE = 0.0065                    # °C per metre: the usual fall of temperature with height
station_fix = None
if STATION:
    try:
        skey, f5 = pix_key("era5", "station"), pix_key("era5", "field")
        def cell_height(key):
            """Average ground height (NASADEM) under an ERA5-Land pixel, close to the height ERA5-Land works at."""
            lon, lat = (float(x) for x in key.split(","))
            box = ee.Geometry.Rectangle([lon - 0.05, lat - 0.05, lon + 0.05, lat + 0.05])
            return with_retries(lambda: ee.Image("NASA/NASADEM_HGT/001").select("elevation")
                                .reduceRegion(ee.Reducer.mean(), box, 90).getInfo())["elevation"]
        z_cs, z_cf = cell_height(skey), cell_height(f5)
        st = pd.DataFrame.from_dict(STATION["months"], orient="index", columns=["tmax", "tmin"]).astype(float)
        st.index = pd.to_datetime(st.index, format="%Y-%m")
        e5s = pd.concat([to_month(daily["era5_tmax"][skey], "mean").rename("e5x"),
                         to_month(daily["era5_tmin"][skey], "mean").rename("e5n")], axis=1)
        both = st.join(e5s, how="inner").dropna()
        # station minus ERA5-Land at the station, moved from the station's height to the field's
        height = LAPSE * (STATION["elev_m"] - z_cs) - LAPSE * (FIELD["elev_m"] - z_cf)
        raw_off, off = {}, {}
        for var, col in (("tmax", "e5x"), ("tmin", "e5n")):
            g = (both[var] - both[col]).groupby(both.index.month)
            mean, count = g.mean(), g.count()
            raw_off[var] = [num(mean[k], 2) if count.get(k, 0) >= 2 else None for k in range(1, 13)]
        good = [k for k in range(12) if raw_off["tmax"][k] is not None and raw_off["tmin"][k] is not None]
        check(len(good) >= 10, f"Station {STATION['name']}: monthly differences from ERA5-Land for {len(good)} of 12 months "
              f"({len(both)} months of data)", fail=f"Station {STATION['name']}: only {len(good)} months with 2+ years of data, so no correction")
        if len(good) >= 10:
            for var in ("tmax", "tmin"):
                avg = float(np.mean([raw_off[var][k] for k in good]))
                off[var] = [num((raw_off[var][k] if k in good else avg) + height, 2) for k in range(12)]
            big = max(abs(v) for var in off for v in off[var])
            check(big <= 4, f"Station correction at the field: daytime highs {np.mean(off['tmax']):+.1f} °C, night lows "
                  f"{np.mean(off['tmin']):+.1f} °C on average (height adjustment {height:+.1f} °C)", warn=True,
                  fail=f"Station correction is up to {big:.1f} °C in some month: check the station file before using it")
            tx = daily["era5_tmax"][f5] + np.array(off["tmax"])[daily["era5_tmax"].index.month.to_numpy() - 1]
            tn = daily["era5_tmin"][f5] + np.array(off["tmin"])[daily["era5_tmin"].index.month.to_numpy() - 1]
            Tc = temp_seasons(tx, tn)
            TKEYS = ["hot_days_flower", "tmax_flower", "frost_days_winter", "late_frost_days"]
            station_fix = {
                "what": "ERA5-Land temperature at the demo field corrected month by month with a nearby weather station "
                        "(station minus ERA5-Land, moved to the field's height at 6.5 °C per km), then the field's heat and "
                        "frost numbers worked out again. Other squares are not corrected.",
                "station": {k: STATION.get(k) for k in ("name", "network", "lat", "lon", "elev_m", "source", "cite")},
                "months_used": len(both), "period": [f"{both.index.min():%Y-%m}", f"{both.index.max():%Y-%m}"],
                "station_minus_era5_c": raw_off, "height_adjust_c": num(height, 2),
                "era5_cell_height_m": {"station": num(z_cs, 0), "field": num(z_cf, 0)}, "field_height_m": FIELD["elev_m"],
                "offset_c": off, "offset_shape": "12 months (Jan-Dec), °C added to ERA5-Land at the field",
                "typical": {k: typical_of(Tc, k, ND[k]) for k in TKEYS},
                "seasons": [{"season": s, **{k: num(Tc[s][k], ND[k]) for k in TKEYS}} for s in SEASONS if s in Tc],
                "note": "Station values were used only to work out the offsets; they are not in this repository.",
            }
            SOURCES["basilicata_stations"] = {
                "name": f"Weather station {STATION['name']} ({STATION.get('network', '')})", "ee_id": "-", "res_m": None,
                "licence": "Free for non-commercial study and research, with citation; not redistributed (used only for the correction)",
                "cite": STATION.get("cite", "")}
            print("Demo field, typical season, ERA5-Land as is -> station-corrected:",
                  {k: (typical_of(T[f5], k, ND[k])[1], station_fix["typical"][k][1]) for k in TKEYS})
    except Exception as e:
        station_fix = None
        check(False, "Station correction", warn=True, fail=f"Station correction skipped ({type(e).__name__}: {str(e)[:100]})")

src_tables = {s: R.get((s, pix_key(s, "field")), {}) for s in PROJ}
src_tables["gsmap"] = monthly_rain_seasons(gsmap_month)
if power is not None:
    src_tables["power"] = rain_seasons(power["PRECTOTCORR"])
    power_temp = temp_seasons(power["T2M_MAX"], power["T2M_MIN"])
# Compare the sources only over seasons that all of them cover (lesson from the NSW practice run)
common = sorted(set.intersection(*[set(t) for t in src_tables.values() if t])) if any(src_tables.values()) else []
common_by_source = {src: num(np.mean([t[y]["rain_year"] for y in common]), 0) for src, t in src_tables.items() if t and common}

season_rows = []
for s in SEASONS:
    row = {"season": s}
    row.update({k: num(f_seasons[s].get(k), ND[k]) for k in ND})
    row["rain_season_by_source"] = {src: num(t[s]["rain_season"], 0) for src, t in src_tables.items() if s in t}
    row["rain_median_of"] = [src for src in PROJ if s in src_tables.get(src, {})]      # main sources behind this season's rain
    if power is not None and s in power_temp:
        row["hot_days_flower_power"] = power_temp[s]["hot_days_flower"]
    season_rows.append(row)

months = pd.date_range(START, TODAY.replace(day=1).isoformat(), freq="MS")
fkeys = {s: pix_key(s, "field") for s in PROJ}
monthly = {"months": [m.strftime("%Y-%m") for m in months]}
for s in PROJ:
    monthly[f"rain_{s}"] = [num(v, 0) for v in M[(s, fkeys[s])].reindex(months)]
monthly["rain_gsmap"] = [num(v, 0) for v in gsmap_month.reindex(months)]
monthly["tmax_era5"] = [num(v, 1) for v in to_month(daily["era5_tmax"][fkeys["era5"]], "mean").reindex(months)]
monthly["tmin_era5"] = [num(v, 1) for v in to_month(daily["era5_tmin"][fkeys["era5"]], "mean").reindex(months)]
if power is not None:
    monthly["rain_power"] = [num(v, 0) for v in to_month(power["PRECTOTCORR"], "sum").reindex(months)]
    monthly["tmax_power"] = [num(v, 1) for v in to_month(power["T2M_MAX"], "mean").reindex(months)]
    monthly["tmin_power"] = [num(v, 1) for v in to_month(power["T2M_MIN"], "mean").reindex(months)]

typ = {k: typical_of(f_seasons, k, nd) for k, nd in ND.items()}
vals = [common_by_source[s] for s in PROJ if common_by_source.get(s)]       # the three sources behind the median
spread = (max(vals) / min(vals) - 1) if len(vals) >= 2 else float("nan")
weather = {
    "made_by": f"{NOTEBOOK} ({RUN_AT})", "field_key": FIELD_KEY,
    "about": "Rain = median of CHIRPS, IMERG and ERA5-Land (GSMaP and POWER shown for comparison). Temperature = ERA5-Land "
             f"(POWER for comparison). Hot day: {HOT:.0f} °C or more; frost night: {FROST:.0f} °C or less; flowering window "
             "15 April to 15 June; autumn = October-November; season = October-May; spring = March-May; days counted from 1 September.",
    "baseline": f"{BASELINE[0]}-{BASELINE[1]}",
    "calendar": f_clim, "typical": typ, "seasons": season_rows, "monthly": monthly,
    "station_corrected": station_fix,
    "agreement": {"rain_year_by_source": f_by_source, "common_seasons": [common[0], common[-1]] if common else None,
                  "rain_year_common_seasons": common_by_source, "spread": num(spread, 2),
                  "removed_months": removed,
                  "note": "rain_year_by_source = typical September-August rain over the normal years (main three sources); "
                          "rain_year_common_seasons = average over the seasons every source covers; spread = largest / smallest - 1 "
                          "for CHIRPS, IMERG and ERA5-Land; removed_months = GSMaP or POWER months dropped as impossible"},
    "sources": ["era5_land", "chirps", "imerg", "gsmap", "power"] + (["basilicata_stations"] if station_fix else []),
}

# Checks
gaps = [s for s in range(common[0], common[-1] + 1) if s not in common] if common else []
span = (f"{len(common)} seasons {common[0]}-{common[-1]}" + (f" (not {', '.join(map(str, gaps))})" if gaps else "")) if common else "no common seasons"
check(spread <= 0.3, f"CHIRPS, IMERG and ERA5-Land agree within 30% at the field over {span} (spread {spread:.0%}); "
                    f"all five: {common_by_source}", warn=True)
normal_season = typ["rain_season"][1]
r24 = next((r["rain_season"] for r in season_rows if r["season"] == 2024), None)
r26 = next((r["rain_season"] for r in season_rows if r["season"] == 2026), None)
if r24 is not None and normal_season:
    check(r24 < normal_season, f"Drought test: October-May rain at the field in season 2024 = {r24} mm (normal {normal_season} mm)", warn=True)
if r26 is not None and normal_season:
    check(r26 > normal_season, f"Wet-year test: October-May rain in season 2026 = {r26} mm (normal {normal_season} mm)", warn=True)
check(typ["hot_days_flower"][1] is not None and 0 <= typ["hot_days_flower"][1] <= 40,
      f"Hot days (≥{HOT:.0f} °C) at flowering, typical season: {typ['hot_days_flower']}")
if power is not None:
    e5 = to_month(daily["era5_tmax"][fkeys["era5"]], "mean")
    pw = to_month(power["T2M_MAX"], "mean")
    diff = float((e5 - pw).dropna().mean())
    check(abs(diff) < 3, f"ERA5-Land and POWER daytime highs agree at the field: mean difference {diff:+.1f} °C", warn=True)

## 7. Write the files

In [ ]:
INDICATORS = {
    "rain_month": {"what": "rain in each month, mm (median of CHIRPS, IMERG, ERA5-Land)", "shape": "12 months x [low, typical, high]"},
    "tmax_month": {"what": "average daytime high in each month, °C (ERA5-Land)", "shape": "12 months x [low, typical, high]"},
    "tmin_month": {"what": "average night-time low in each month, °C (ERA5-Land)", "shape": "12 months x [low, typical, high]"},
    "hot30_month": {"what": f"days at {HOT:.0f} °C or more in each month (ERA5-Land)", "shape": "12 months x [low, typical, high]"},
    "frost_month": {"what": f"frost nights ({FROST:.0f} °C or less) in each month (ERA5-Land)", "shape": "12 months x [low, typical, high]"},
    "rain_autumn": {"what": "October-November rain, mm (sowing time)", "shape": "[low, typical, high]"},
    "rain_season": {"what": "October-May rain, mm (the durum season)", "shape": "[low, typical, high]"},
    "rain_spring": {"what": "March-May rain, mm (grain filling)", "shape": "[low, typical, high]"},
    "rain_year": {"what": "September-August rain, mm", "shape": "[low, typical, high]"},
    "rain_onset_day": {"what": f"day the autumn rain starts ({ONSET_MM:.0f} mm in 3 days), counted from 1 September (median of CHIRPS and ERA5-Land)", "shape": "[low, typical, high]"},
    "dry_spell_spring": {"what": f"longest run of dry days (under {WET_DAY:.0f} mm) in March-May", "shape": "[low, typical, high]"},
    "hot_days_flower": {"what": f"days at {HOT:.0f} °C or more from 15 April to 15 June (durum flowering and grain fill)", "shape": "[low, typical, high]"},
    "tmax_flower": {"what": "average daytime high from 15 April to 15 June, °C", "shape": "[low, typical, high]"},
    "frost_days_winter": {"what": "frost nights from December to February", "shape": "[low, typical, high]"},
    "late_frost_days": {"what": "frost nights in March and April", "shape": "[low, typical, high]"},
    "rain_year_by_source": {"what": "typical September-August rain from each source, mm: shows how far the sources agree", "shape": "{source: mm}"},
    "rain_season_by_season": {"what": "October-May rain of each season, mm", "shape": "one value per season in meta.seasons"},
    "rain_autumn_by_season": {"what": "October-November rain of each season, mm", "shape": "one value per season in meta.seasons"},
    "hot_days_flower_by_season": {"what": "hot days at flowering in each season", "shape": "one value per season in meta.seasons"},
}
cells_doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                      "keys": "cells maps each farmland square id (from grid.geojson) to a key in records: "
                              "the values for a square are records[cells[id]]",
                      "low_typical_high": f"10th percentile, median and 90th percentile of the seasons {BASELINE[0]}-{BASELINE[1]}",
                      "baseline": f"{BASELINE[0]}-{BASELINE[1]}", "seasons": SEASONS,
                      "season": "named by harvest year: season 2024 = September 2023 to August 2024",
                      "rain_sources_by_season": {str(s): [src for src in PROJ if s in src_tables.get(src, {})] for s in SEASONS
                                                 if len([src for src in PROJ if s in src_tables.get(src, {})]) < len(PROJ)},
                      "how": "each source's value is copied from the pixel containing the square's centre (5-11 km pixels; "
                             "squares whose ERA5-Land pixel is sea use the nearest land pixel), "
                             "so squares under the same pixels share a record; rain is the median of the sources",
                      "indicators": INDICATORS, "sources": {k: SOURCES[k] for k in ("era5_land", "chirps", "imerg")}},
             "cells": cells_out, "records": records}

OUTPUTS = ["field_weather.json", "cells_weather.json"]
check(write_field_section("weather", {"weather": weather}, SOURCES) < 5, "field_weather.json written")
mb = write_json(DATA / "cells_weather.json", cells_doc)
check(mb < 5, f"cells_weather.json written: {mb:.2f} MB")

## 8. One chart to explain to a farmer
October–May rain at the demo field each season, from each source; the line is the median of the main three.

In [ ]:
import matplotlib.pyplot as plt

x = np.array([r["season"] for r in season_rows])
def by_source(src):
    return np.array([np.nan if r["rain_season_by_source"].get(src) is None else r["rain_season_by_source"][src]
                     for r in season_rows], dtype=float)

STYLE = {"chirps": ("CHIRPS", "#6a9f58", "-"), "imerg": ("IMERG", "#d18c3e", "-"), "era5": ("ERA5-Land", "#4a6fa5", "-"),
         "gsmap": ("GSMaP (comparison only)", "#9b59b6", "--"), "power": ("NASA POWER (comparison only)", "#7f8c8d", "--")}
fig, ax = plt.subplots(figsize=(11, 4.6))
for season, colour, text, ytxt in ((2024, "#c0392b", "2024 drought", 0.98), (2026, "#1f6fb2", "2026 wet", 0.91)):
    ax.axvspan(season - 0.45, season + 0.45, color=colour, alpha=0.10, lw=0)
    ax.text(season, ytxt, text, transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=8, color=colour)
for src, (label, colour, ls) in STYLE.items():
    y = by_source(src)
    if np.isfinite(y).any():
        ax.plot(x, y, color=colour, ls=ls, lw=1.2, marker="o", ms=3, label=label)
median = np.array([np.nan if r["rain_season"] is None else r["rain_season"] for r in season_rows], dtype=float)
ax.plot(x, median, color="black", lw=2.6, marker="o", ms=4, zorder=5, label="Median of CHIRPS, IMERG, ERA5-Land")
if normal_season:
    ax.axhline(normal_season, color="black", ls=":", lw=1, label=f"Normal {BASELINE[0]}-{BASELINE[1]}: {normal_season:.0f} mm")
main = np.concatenate([by_source(s) for s in ("chirps", "imerg", "era5")] + [median])
ax.set_ylim(max(0, np.nanmin(main) - 100), np.nanmax(main) + 150)      # scaled to the main sources, so nothing squashes them
ax.set_xlim(x.min() - 0.6, x.max() + 0.6)
ax.set_xticks(x[::2])
ax.set_xlabel("Season (named by harvest year: 2024 = September 2023 to August 2024)")
ax.set_ylabel("October-May rain (mm)")
ax.set_title("Demo field, Genzano di Lucania: rain in each durum season, by source", loc="left")
ax.legend(frameon=False, ncol=3, fontsize=8, loc="upper left", bbox_to_anchor=(0, -0.18))
ax.grid(alpha=0.3)
plt.savefig(DATA.parent / "rain_field.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Checks, downloads and what to paste back

In [ ]:
print("\n".join(CHECKS))
bad = [c for c in CHECKS if c.startswith("❌")]
print("\nAll essential checks passed." if not bad else f"\n{len(bad)} check(s) failed: paste the lines above into the chat.")
if IN_COLAB:
    from google.colab import files
    for name in OUTPUTS:
        files.download(str(DATA / name))
    print("Downloading", ", ".join(OUTPUTS), "- upload them to the repo's data folder (replace the old ones).")

In [ ]:
print("=== Paste this back to Claude ===")
print(f"03_weather run {RUN_AT}; seasons {SEASONS[0]}-{SEASONS[-1]}; pixels {({s: len(u) for s, u in UNIQUE.items()})}")
print("field typical:", json.dumps(typ))
print("field rain by source (typical Sep-Aug mm):", json.dumps(f_by_source), "| common seasons:", json.dumps(common_by_source), f"spread {spread:.0%}")
print("field seasons 2017-:", json.dumps([{k: r[k] for k in ("season", "rain_autumn", "rain_season", "rain_onset_day", "hot_days_flower", "frost_days_winter")} for r in season_rows if r["season"] >= 2017]))
print("field monthly rain typical:", [m[1] for m in f_clim["rain_month"]])
print("field monthly tmax typical:", [m[1] for m in f_clim["tmax_month"]])
print("\n".join(CHECKS))